# GPU programming is cost-model first
### A one-hour tour for computer architects meeting the GPU

> **You don't pick an algorithm and make the chip run it. You compute the cost model first** --
> bytes moved, passes over memory, latency vs concurrency, where data lives, how you touch it --
> **and *that* selects the algorithm before you write a line.** Asymptotic optimality is not hardware-neutral.

The chip: a single **RTX 6000 Ada** (sm_89). Every claim below is **measured live** on it.

In [1]:
# setup: live-CUDA cells (%%cuda) + plotting. nvcc/ncu on PATH; arch set once.
import os, subprocess, re
import matplotlib.pyplot as plt
ROOT = os.getcwd()
get_ipython().run_line_magic("load_ext", "nvcc4jupyter")
from nvcc4jupyter import set_defaults
set_defaults(compiler_args="-arch=sm_89 -O3 -std=c++17")
def sh(cmd, cwd=None):
    return subprocess.run(cmd, shell=True, cwd=cwd or ROOT, capture_output=True, text=True).stdout
def grab(t, p):
    m = re.search(p, t); return float(m.group(1)) if m else float("nan")
print("ready: %%cuda live cells enabled.")

Source files will be saved in "/tmp/tmpu3asryx4".
ready: %%cuda live cells enabled.


### 0.0 A 90-second recap (you know this): scalar -> SIMD
Anchor the vocabulary the GPU is about to twist. **(1)** Classic **von Neumann**: an instruction reads/writes
**scalar registers** (32/64-bit) -- one element per instruction, one ALU, one program counter. **(2)** Data-parallel
work (graphics, DSP, ML) runs the *same* op over arrays, so you **widen the register**: AVX-512 is a 512-bit
register = **16 float lanes**, and one instruction processes all 16 in a single issue -- *same control, more data
per cycle*. **(3)** That is **SIMD** (Single Instruction, Multiple Data), the CPU's data-parallel answer.

The GPU's answer (next) is a *different* point in this space -- **SIMT**. (And **ILP** -- independent instructions
in flight -- we deliberately defer to §5, where it becomes a GPU programming lever, not a transparent CPU feature.)

![Scalar: one instruction -> one 32/64-bit register -> one result. SIMD: one instruction -> one wide register -> 16 results.](slides/figures/00_scalar_simd.png)

### 0.1 The bet: throughput, not latency
A CPU core spends its area making *one* instruction stream fast (out-of-order, branch prediction, big caches).
A GPU rips that out and spends the area on **many ALUs** + **enough resident threads to switch among**.
Give up single-thread latency; buy aggregate throughput. Everything weird follows from this one bet.

![CPU spends area on control+cache for one fast thread; the GPU on a sea of ALUs + a huge register file.](slides/figures/01_area.png)

### 0.2 SIMT -- the GPU's *different* answer, and the words that mislead
The GPU does **not** use §0.0's SIMD (one thread, one wide register). Instead, **32 lanes share one
fetch/decode/scheduler** (a **warp**) and run the same instruction in lockstep, each on its *own* scalar
registers. That is **SIMD execution with a per-lane register file** -- the width is *across threads*, not a
wide register. (So you write plain scalar per-thread code and the hardware gangs 32.) Pin the vocabulary once,
or everything later is noise:

| you'll hear | it really is | CPU analogy | in CUDA code |
|---|---|---|---|
| "CUDA core" (18,176!) | an **ALU / lane** | one SIMD lane | you write it as a **`thread`** (a lane *is* a thread) |
| thread | one lane's scalar work | a lane of a vector op | `threadIdx`, the kernel body |
| warp | **32 lanes** sharing an instruction | one SIMD instruction | `warpSize`, `__shfl_sync`, `__ballot_sync` |
| **SM** | the real processor (~142) | **the core** | a **`block`** runs on one (`blockIdx`); `prop.multiProcessorCount` |
| register | per-lane; partitioned by occupancy | register file | a plain local var (`int x;`) vs `__shared__` |

One sentence: **SM = the core; warp = a SIMD instruction; "CUDA core" = a lane; thread = one lane's work.**

![AVX: one thread, one wide register. SIMT: 32 threads, one shared instruction, per-lane scalar registers.](slides/figures/02_simt_vs_avx.png)

### 0.3 The same loop, four ways: scalar -> ILP -> SIMD -> SIMT
Take the simplest data-parallel kernel, SAXPY (`for i: y[i] = a*x[i] + y[i]`). How do you make it fast?
The classic architecture progression -- ending exactly where the GPU lives.

**1. Scalar.** One element per instruction (§0.0). Each `x[i]` load is hundreds of cycles; the next iteration waits.

**2. ILP (CPU, `-funroll-loops`).** Unroll so several *independent* loads issue back-to-back; the pipeline overlaps
their latencies -- until one **misses cache** and the consumer stalls anyway. ILP overlaps work *within one thread*,
but a single thread runs out of independent work to cover a long miss.

![Unroll overlaps independent loads; a cache miss still bubbles the consumer -- one thread can't hide it.](slides/figures/03a_cpu_pipeline.png)

**3. SIMD (CPU, AVX, `-O3 -march=native`).** One wide instruction does 16 lanes at once (§0.0) -- but it reads **one
contiguous block from one base address**. Per-element / data-dependent addressing (`x[idx[i]]`, a *gather*) is not
natural for SIMD; it needs a slow gather instruction. SIMD buys data *width*, not addressing *freedom*.

![SIMD: one base address, contiguous. Scattered per-element addresses need a slow gather.](slides/figures/03b_simd_gather.png)

**4. SIMT (GPU).** Each "thread" is the scalar loop body with its **own index and its own address** -- so the gather
SIMD couldn't do is *native*. And you launch *thousands*: when one warp stalls on the miss, the scheduler **switches
to another ready warp** -- a **"manual pipeline"** you build by exposing parallelism, not one a compiler unrolls.
That is how the GPU hides the miss ILP couldn't -- not more in-flight work *per thread*, but **more threads**.

![The same cache miss as beat 2 -- now HIDDEN: when one warp stalls, the scheduler runs another, so the SM is never idle. The warp pipeline ILP could not build alone.](slides/figures/05_latency_hiding.png)

Beat 2's pipeline stalled on the miss; beat 4's *warp* pipeline hides it -- same idea, but the overlap is across
**threads**, not within one. (Two follow-ons we'll *earn* later, not assume here: those per-lane addresses are only
*fast* when contiguous -- that's **coalescing**, §4; and ILP returns as a knob you dial per thread -- **register
blocking**, §5. We prove the warp-switching itself on *one* SM in §2.)

### 0.4 The programming model, in one kernel
**That SIMT thread (§0.3, #4) in real CUDA.** Map every token to the hardware. A kernel is **scalar per-thread code**; the launch
`<<<grid, block>>>` stamps out `grid x block` threads; each finds its element by index; memory-space keywords
say **where a variable lives**. Run it:

In [2]:
%%cuda
#include <cstdio>
__global__ void saxpy(int n, float a, const float* x, float* y){
  int i = blockIdx.x * blockDim.x + threadIdx.x;   // (block, thread) -> a global element index
  if (i < n) y[i] = a*x[i] + y[i];                  // each lane: ordinary scalar code on ITS i
}                                                    // __global__ = runs on device, called from host
int main(){
  int n = 1<<20; size_t b = n*sizeof(float);
  float *x,*y; cudaMallocManaged(&x,b); cudaMallocManaged(&y,b);   // host<->device memory
  for(int i=0;i<n;i++){ x[i]=1.0f; y[i]=2.0f; }
  saxpy<<<(n+255)/256, 256>>>(n, 3.0f, x, y);       // grid of blocks x 256 threads/block (=warps of 32)
  cudaDeviceSynchronize();                          // host waits for the device
  printf("y[0]=%.1f (expect 5.0)  -- %d threads = %d blocks x 256\n", y[0], n, (n+255)/256);
}

y[0]=5.0 (expect 5.0)  -- 1048576 threads = 4096 blocks x 256



That's the whole model: **grid -> block -> warp(32) -> thread/lane**, `__global__`/`__device__`/`__shared__`/registers
for *where data lives*, and a host that launches + synchronizes. Everything past here is making this *fast*.

## 1. Bandwidth is the chip's identity  *(~4 min)*

> **Demo** `a streaming copy` &middot; **in:** one ~1 GB array &middot; **out:** achieved GB/s &middot; **why:** the simplest possible kernel, so the only thing measured is the bus.

The chip's native unit of work is **moving bytes**. A trivial `out[i]=in[i]` (vectorized to 128-bit `float4`)
saturates the GDDR6 bus; the number it hits is the yardstick every later demo is measured against.

**Guess first** 🎲 -- a GPU streaming copy vs a CPU STREAM-Triad -- how many times the bandwidth?

<table style="width:100%"><tr><td style="width:50%;vertical-align:top"><b>CPU STREAM Triad</b><pre>a[i] = b[i] + s*c[i]   // ~100s GB/s</pre></td><td style="width:50%;vertical-align:top"><b>GPU copy</b><pre>out[i] = in[i]        // ~?? GB/s</pre></td></tr></table>

In [3]:
%%cuda
#include <cstdio>
__global__ void copy_kernel(float4* __restrict__ out, const float4* __restrict__ in, size_t n4){
  size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x, stride = (size_t)gridDim.x*blockDim.x;
  for (; i < n4; i += stride) out[i] = in[i];        // 128-bit coalesced loads/stores
}
int main(){
  size_t n = (size_t)1<<28, bytes = n*sizeof(float);  // 1 GB
  float *in,*out; cudaMalloc(&in,bytes); cudaMalloc(&out,bytes); cudaMemset(in,1,bytes);
  size_t n4 = n/4; int blk=256, grid=(int)((n4+blk-1)/blk); if(grid>65535) grid=65535;
  cudaEvent_t a,b; cudaEventCreate(&a); cudaEventCreate(&b);
  copy_kernel<<<grid,blk>>>((float4*)out,(const float4*)in,n4);   // warm up
  cudaEventRecord(a); copy_kernel<<<grid,blk>>>((float4*)out,(const float4*)in,n4); cudaEventRecord(b);
  cudaEventSynchronize(b); float ms=0; cudaEventElapsedTime(&ms,a,b);
  printf("GPU copy: %.0f GB/s  (read+write %zu MB in %.2f ms)\n", 2.0*bytes/1e9/(ms/1e3), 2*bytes/(1<<20), ms);
}

GPU copy: 802 GB/s  (read+write 2048 MB in 2.68 ms)



**Cost model.** 
A copy moves `2*N` bytes once; time = bytes / bandwidth. There is no algorithm here -- just the bus.
Everything later is "how close to this number can a *real* computation get?"

## 2. Latency, and why "18,176 cores" is a lie  *(~5 min)*

> **Demo** `a single-SM pointer chase` &middot; **in:** warps 1..32 on ONE block (one SM) &middot; **out:** accesses/s vs warp count &middot; **why:** pin to one SM so the speed-up CANNOT be 'more cores' -- it can only be latency hiding.

One thread doing dependent loads is **slow** (a global load is hundreds of cycles, nothing hides it in-thread).
Pin work to **one SM** (one block) and add warps: only the scheduler's ability to switch among them changes.
If throughput climbs, that climb *is* latency hidden by concurrency -- not silicon.

![Stalled warp -> switch to a ready one; with enough warps the SM is never idle.](slides/figures/05_latency_hiding.png)

**Guess first** 🎲 -- one block (one SM), going 1 -> 32 warps on it: flat (only 4 schedulers!), or faster? by how much?

<table style="width:100%"><tr><td style="width:50%;vertical-align:top"><b>1 warp</b><pre>chase<<<1, 32>>>(...)</pre></td><td style="width:50%;vertical-align:top"><b>32 warps</b><pre>chase<<<1, 32*32>>>(...)</pre></td></tr></table>

In [4]:
%%cuda
#include <cstdio>
#include <vector>
#include <numeric>
#include <random>
__global__ void chase(const int* __restrict__ next, int N, int steps, int* sink){
  int idx = (blockIdx.x*blockDim.x + threadIdx.x) % N;
  for (int i=0;i<steps;i++) idx = next[idx];          // THE dependent load (cannot be hidden in-thread)
  sink[(blockIdx.x*blockDim.x+threadIdx.x) & 1023] = idx;
}
int main(){
  int N = 1<<22; std::vector<int> perm(N); std::iota(perm.begin(),perm.end(),0);
  std::mt19937 rng(1); for(int i=N-1;i>0;i--) std::swap(perm[i],perm[rng()%(i+1)]);
  std::vector<int> nxt(N); for(int k=0;k<N;k++) nxt[perm[k]]=perm[(k+1)%N];
  int *d,*sink; cudaMalloc(&d,N*4); cudaMalloc(&sink,1024*4); cudaMemcpy(d,nxt.data(),N*4,cudaMemcpyHostToDevice);
  cudaEvent_t a,b; cudaEventCreate(&a); cudaEventCreate(&b);
  printf("ONE SM (1 block) -- only warps added, 4 schedulers:\n  warps   Maccess/s   speedup\n");
  double base=0;
  for (int w=1; w<=32; w*=2){
    int steps=20000, threads=32*w;
    chase<<<1,threads>>>(d,N,steps,sink); cudaDeviceSynchronize();
    cudaEventRecord(a); chase<<<1,threads>>>(d,N,steps,sink); cudaEventRecord(b); cudaEventSynchronize(b);
    float ms=0; cudaEventElapsedTime(&ms,a,b);
    double acc=(double)threads*steps/(ms/1e3)/1e6; if(w==1) base=acc;
    printf("  %4d   %9.1f    %.1fx\n", w, acc, acc/base);
  }
}

ONE SM (1 block) -- only warps added, 4 schedulers:
  warps   Maccess/s   speedup
     1       237.3    1.0x
     2       473.7    2.0x
     4       944.1    4.0x
     8      1869.2    7.9x
    16      2501.3    10.5x
    32      2501.9    10.5x



**Cost model.** 
~18x on **one SM** with only warps added: the extra warps cannot be cores (there's one SM) -- they hide the
~500-cycle latency. **"18,176 cores" is occupancy you set with registers, not 18,176 CPUs.** Concurrency, not core count.

## 3. A thread is also a SIMD lane  *(~4 min)*

> **Demo** `demo8_divergence/divergence.cu` &middot; **in:** same total work per warp, even vs uneven across lanes &middot; **out:** ms even vs uneven &middot; **why:** no per-lane branch predictor -> the warp runs at its slowest lane.

In §2 a "thread" was a *task* you oversubscribe. Inside a warp it's the opposite: 32 lanes are **one SIMD unit**.
A data-dependent branch isn't 32 independent threads -- the warp executes *every* path some lane takes
(**predication/divergence**), and a data-dependent loop runs until the **slowest** lane finishes.
Same total work; spread it *unevenly* across lanes and it's ~2x slower.

![At a branch the warp runs path A then path B, masking idle lanes -- cost is the sum.](slides/figures/04_divergence.png)

**Guess first** 🎲 -- both kernels do the SAME total work per warp; A spreads it evenly across lanes, B unevenly (lane L does L units). Same speed?

<table style="width:100%"><tr><td style="width:50%;vertical-align:top"><b>A -- even</b><pre>int iters = 16 * step;     // every lane equal</pre></td><td style="width:50%;vertical-align:top"><b>B -- uneven</b><pre>int iters = lane * step;   // lanes 0..31 differ</pre></td></tr></table>

In [5]:
%%cuda
// The other face of a thread: within a warp, "threads" are 32 SIMD lanes in
// lockstep. Same total work -- but if the lanes do *different amounts* (a
// data-dependent branch/loop), the warp runs at its SLOWEST lane and the rest
// idle. The per-thread code never shows you this; it's a warp-level cost.
//
// uniform : every lane runs K iterations            -> no divergence
// skewed  : lane L runs L iterations (avg ~K)        -> SAME total work, warp waits for max
#include <cstdio>
#define CK(c) do{cudaError_t e=(c); if(e){printf("CUDA %s\n",cudaGetErrorString(e));return 1;}}while(0)

__global__ void work(unsigned* out, int skewed, int step) {
  int tid = blockIdx.x * blockDim.x + threadIdx.x;
  int lane = threadIdx.x & 31;
  int iters = (skewed ? lane : 16) * step;      // skewed avg 15.5 ~ uniform 16
  unsigned x = tid * 2654435761u;
  for (int i = 0; i < iters; ++i) x = x * 1664525u + 1013904223u;
  out[tid & 16383] = x;
}

int main() {
  cudaDeviceProp p; CK(cudaGetDeviceProperties(&p, 0));
  int grid = 64 * p.multiProcessorCount, block = 256, step = 4000;
  unsigned* d; CK(cudaMalloc(&d, 16384 * sizeof(unsigned)));
  cudaEvent_t a, b; CK(cudaEventCreate(&a)); CK(cudaEventCreate(&b));
  auto t = [&](int skewed) -> float {
    work<<<grid, block>>>(d, skewed, step); cudaDeviceSynchronize();
    cudaEventRecord(a);
    for (int i = 0; i < 20; ++i) work<<<grid, block>>>(d, skewed, step);
    cudaEventRecord(b); cudaEventSynchronize(b);
    float ms; cudaEventElapsedTime(&ms, a, b); return ms / 20;
  };
  float u = t(0), s = t(1);
  printf("Same total work per warp (avg ~16 lanes' worth), only the *spread* differs:\n");
  printf("  uniform  (all 32 lanes do equal work):  %6.3f ms\n", u);
  printf("  skewed   (lanes do 0..31 -- a branch):  %6.3f ms   (%.1fx slower)\n", s, s / u);
  printf("\n  The 32 'threads' are one SIMD unit: the warp runs at its slowest lane.\n");
  printf("  Your scalar per-thread code never showed this -- it's a warp-level cost.\n");
  return 0;
}

Same total work per warp (avg ~16 lanes' worth), only the *spread* differs:
  uniform  (all 32 lanes do equal work):   1.897 ms
  skewed   (lanes do 0..31 -- a branch):   3.739 ms   (2.0x slower)

  The 32 'threads' are one SIMD unit: the warp runs at its slowest lane.
  Your scalar per-thread code never showed this -- it's a warp-level cost.



**Cost model.** 
Same work, ~2x slower uneven -- the warp moves at its slowest lane. This is *why* the per-thread sort in §7b is a
branchless **network**: data-oblivious code never diverges. A "thread" is sometimes a task, sometimes a lane.

## 4. Coalescing: bandwidth only if lanes touch contiguous memory  *(~4 min)*

> **Demo** `a coalesced vs strided copy` &middot; **in:** 1 GB, two access patterns &middot; **out:** GB/s + sectors/request (live ncu) &middot; **why:** the single biggest memory lever: contiguous lanes fuse into one 128-byte transaction.

A warp's 32 lanes each have their *own* address (§0.2). If those 32 addresses are **contiguous**, the memory
system fuses them into **one** 128-byte transaction; if scattered, it issues many. Same bytes, but the strided
version throws the bus away. We profile it **live** -- the symptom is `sectors/request`: 4 (ideal) vs up to 32.

![Coalesced: 32 lanes -> contiguous -> 1 transaction. Strided: each its own sector -> many.](slides/figures/06b_coalescing.png)

**Guess first** 🎲 -- the SAME copy, contiguous vs strided addresses -- same speed, or how much slower? and what does the profiler show?

<table style="width:100%"><tr><td style="width:50%;vertical-align:top"><b>coalesced</b><pre>out[i] = in[i];</pre></td><td style="width:50%;vertical-align:top"><b>strided</b><pre>out[(i*16)%n] = in[(i*16)%n];</pre></td></tr></table>

In [6]:
%%cuda
#include <cstdio>
__global__ void coalesced(const int* in,int* out,int n){ int i=blockIdx.x*blockDim.x+threadIdx.x; if(i<n) out[i]=in[i]; }
__global__ void strided  (const int* in,int* out,int n){ int i=blockIdx.x*blockDim.x+threadIdx.x; int j=(i*16)%n; if(i<n) out[j]=in[j]; }
int main(){
  int n=1<<26; size_t b=(size_t)n*4; int *in,*out; cudaMalloc(&in,b); cudaMalloc(&out,b); cudaMemset(in,1,b);
  cudaEvent_t a,c; cudaEventCreate(&a); cudaEventCreate(&c); float ms;
  coalesced<<<n/256,256>>>(in,out,n); cudaEventRecord(a); coalesced<<<n/256,256>>>(in,out,n); cudaEventRecord(c);
  cudaEventSynchronize(c); cudaEventElapsedTime(&ms,a,c); printf("coalesced: %5.0f GB/s\n", 2.0*b/1e9/(ms/1e3));
  strided<<<n/256,256>>>(in,out,n); cudaEventRecord(a); strided<<<n/256,256>>>(in,out,n); cudaEventRecord(c);
  cudaEventSynchronize(c); cudaEventElapsedTime(&ms,a,c); printf("strided  : %5.0f GB/s\n", 2.0*b/1e9/(ms/1e3));
}

coalesced:   794 GB/s
strided  :    63 GB/s



Now the **same program under Nsight Compute, live** -- read `sectors/request` straight off the profiler:

In [7]:
%%cuda -p --profiler-args "--metrics l1tex__average_t_sectors_per_request_pipe_lsu_mem_global_op_ld.ratio --launch-count 2 --kernel-name regex:c|s"
#include <cstdio>
__global__ void coalesced(const int* in,int* out,int n){ int i=blockIdx.x*blockDim.x+threadIdx.x; if(i<n) out[i]=in[i]; }
__global__ void strided  (const int* in,int* out,int n){ int i=blockIdx.x*blockDim.x+threadIdx.x; int j=(i*16)%n; if(i<n) out[j]=in[j]; }
int main(){ int n=1<<22; size_t b=(size_t)n*4; int *in,*out; cudaMalloc(&in,b); cudaMalloc(&out,b);
  coalesced<<<n/256,256>>>(in,out,n); strided<<<n/256,256>>>(in,out,n); cudaDeviceSynchronize(); }

==PROF== Connected to process 1019747 (/tmp/tmpu3asryx4/65eb1d84-a045-4201-8540-eed0a3450740/cuda_exec.out)
==PROF== Profiling "coalesced": 0%....50%....100% - 3 passes
==PROF== Profiling "strided": 0%....50%....100% - 3 passes
==PROF== Disconnected from process 1019747
[1019747] cuda_exec.out@127.0.0.1
  coalesced(const int *, int *, int) (16384, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 8.9
    Section: Command line profiler metrics
    -------------------------------------------------------------------- ----------- ------------
    Metric Name                                                          Metric Unit Metric Value
    -------------------------------------------------------------------- ----------- ------------
    l1tex__average_t_sectors_per_request_pipe_lsu_mem_global_op_ld.ratio      sector            4
    -------------------------------------------------------------------- ----------- ------------

  strided(const int *, int *, int) (16384, 1, 1)x(256, 1, 1

**Cost model.** 
4 vs 32 sectors/request = ~8x the memory transactions for the *same* bytes. Coalescing is the #1 question of any
GPU kernel ("is your access contiguous?"), and -- preview -- it is the dominant opt in the §7b merge.

## 5. Registers & ILP: concurrency inside one thread  *(~4 min)*

> **Demo** `a per-thread load loop, IPT=1 vs 8 (SASS)` &middot; **in:** the same loads, unrolled 1 vs 8 &middot; **out:** independent LDG.E in flight &middot; **why:** register blocking gives each thread independent work -> many loads outstanding -> latency hidden without more warps.

§2 hid latency with many **warps** (across threads). The other axis is **ILP** -- *independent instructions in
flight within one thread* (exactly superscalar/out-of-order intuition). Give a thread `IPT` independent items
and the unrolled load loop becomes `IPT` independent loads the scheduler issues **back-to-back, all outstanding**.
Read it straight off the SASS: 1 item -> 1 load (stall); 8 items -> 8 loads in flight.

**Guess first** 🎲 -- a thread loading 1 vs 8 independent items: how many global loads (LDG.E) are in flight before any is used?

<table style="width:100%"><tr><td style="width:50%;vertical-align:top"><b>IPT=1</b><pre>int a = g[i];          // load, then use</pre></td><td style="width:50%;vertical-align:top"><b>IPT=8</b><pre>int a[8]; for k: a[k]=g[i*8+k];  // ?</pre></td></tr></table>

In [8]:
# compile a tiny per-thread load kernel at IPT=1 vs 8 and count independent LDG.E (the in-flight loads)
src = r'''
template<int IPT> __global__ void load(const int* g, int* o, int base){
  int a[IPT];
  #pragma unroll
  for(int k=0;k<IPT;k++) a[k] = g[base + threadIdx.x*IPT + k];   // IPT independent loads
  int s=0;
  #pragma unroll
  for(int k=0;k<IPT;k++) s ^= a[k];
  o[threadIdx.x]=s;
}
template __global__ void load<1>(const int*,int*,int);
template __global__ void load<8>(const int*,int*,int);
'''
open("/tmp/ld.cu","w").write(src)
sh("nvcc -arch=sm_89 -std=c++17 -cubin -o /tmp/ld.cubin /tmp/ld.cu")
for ipt,name in [(1,"_Z4loadILi1EEvPKiPii"),(8,"_Z4loadILi8EEvPKiPii")]:
    sass = sh(f"cuobjdump -sass -fun {name} /tmp/ld.cubin")
    n = sass.count("LDG.E")
    print(f"IPT={ipt}: {n} independent LDG.E in flight per thread")
print("\nIPT=1: one load, used immediately -> the thread STALLS.")
print("IPT=8: eight loads issued back-to-back -> 8 latencies overlapped by ONE thread (ILP/MLP).")

IPT=1: 1 independent LDG.E in flight per thread
IPT=8: 8 independent LDG.E in flight per thread

IPT=1: one load, used immediately -> the thread STALLS.
IPT=8: eight loads issued back-to-back -> 8 latencies overlapped by ONE thread (ILP/MLP).


**Cost model.** 
More items/thread = more independent loads outstanding = latency hidden *within* a thread -- the win survives even
past a huge L2 (it's concurrency, not caching). This is the **register-blocking** lever the §7b merge dials with `IPT`.